# Fundamentals of Data Science - Network Science Lab

# Lab Session 06 - Random Graph Models

In [ ]:
import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path

In [ ]:
def degree_distribution(graph, mode="all"):
  # This function computes the degree distribution of a (directed) graph.
  degrees = graph.degree(mode=mode)
  degree_frequency = pd.Series(degrees).value_counts()
  degree_frequency = degree_frequency.sort_index()
  full_range = range(min(graph.degree()+[1]), max(graph.degree())+1)
  degree_frequency = degree_frequency.reindex(full_range, fill_value=0)

  degree_distribution = degree_frequency / degree_frequency.sum()
  return degree_distribution

Random graph models come in handy to construct ensembles, which represent probability distributions over possible network configurations.

In fact, one can assess the statistical significance of a property observed in a real network by comparing it against synthetic random graphs.

## The Erdős-Rényi Model

The Erdős-Rényi model is an ensemble $G(n, p)$ that fixes the number of nodes $n$ and the independent probability $p$ of having an edge between any two nodes.

In [ ]:
random.seed(10) # For reproducibility.

# Create four networks using the Erdős-Rényi model, choosing n = 25 and p = 0.1.
er1 = ig.Graph.Erdos_Renyi(n = 25, p = 0.1)
er2 = ig.Graph.Erdos_Renyi(n = 25, p = 0.1)
er3 = ig.Graph.Erdos_Renyi(n = 25, p = 0.1)
er4 = ig.Graph.Erdos_Renyi(n = 25, p = 0.1)

# Plot the generated networks.
fig, ax = plt.subplots(2,2,figsize=(9, 8))

ig.plot(
    er1,
    target=ax[0,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,0].set_title("First Graph")

ig.plot(
    er2,
    target=ax[0,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,1].set_title("Second Graph")

ig.plot(
    er3,
    target=ax[1,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,0].set_title("Third Graph")

ig.plot(
    er4,
    target=ax[1,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,1].set_title("Fourth Graph")

plt.tight_layout()
plt.show()

Since edges may or may not be added through independent and identically distributed Bernoulli trials, it is possible to discover that the edges in an Erdős-Rényi network follow a binomial distribution defined by
$$\mathbb{P}(m) = p^m(1 - p)^{\binom{n}{2} - m} \Rightarrow \langle m \rangle = \binom{n}{2}p$$

This phenomenon implies that the degree distribution of an Erdős-Rényi network is also binomial, with
$$p_k = \binom{n - 1}{k}p^k(1 - p)^{n - 1 - k} \Rightarrow \langle k \rangle = (n - 1)p$$

However, for many real networks, where $n \to +\infty$, the degree distribution tends to be approximated with a Poisson distribution with rate $\lambda = \langle k \rangle$, resulting in
$$p_k = \frac{e^{-\langle k \rangle}\langle k \rangle^k}{k!}$$

However, since the binomial and Poisson distributions lack heavy tails, the Erdős-Rényi model fails to reproduce the heavy-tailed degree distributions observed in real networks.

In [ ]:
def degree_distribution(graph, mode="all"):
  # This function computes the degree distribution of a (directed) graph.
  degrees = graph.degree(mode=mode)
  degree_frequency = pd.Series(degrees).value_counts()
  degree_frequency = degree_frequency.sort_index()
  full_range = range(min(graph.degree()+[1]), max(graph.degree())+1)
  degree_frequency = degree_frequency.reindex(full_range, fill_value=0)

  degree_distribution = degree_frequency / degree_frequency.sum()
  return degree_distribution

# Analyse the degree distribution of an Erdős-Rényi random network.
g_er = ig.Graph.Erdos_Renyi(n = 100, p = 0.1)
g_er.summary()

fig, ax = plt.subplots(figsize=(5, 4))

# Plot the distribution using a frequency scatterplot.
ax.scatter(degree_distribution(g_er).index,
degree_distribution(g_er).values, color = "red")
ax.set_title("Frequency of Elements (Scatter Plot)")
ax.set_xlabel("Degree")
ax.set_ylabel("Frequency")
ax.set_box_aspect(1)

In [ ]:
random.seed(10) # For reproducibility.

# Extract information about the degree and edge distributions of 100 Erdős-Rényi random networks with n = 100 and p = 0.1.
er_degree_values = []
er_m = []
for i in range(0,100):
  tmp_er = ig.Graph.Erdos_Renyi(n = 100, p = 0.1)
  er_degree_values.append(tmp_er.degree())
  er_m.append(tmp_er.ecount())

# Plot the resulting distributions.
fig, ax = plt.subplots(1, 2, figsize=(9, 4))#

# Degree distribution.
ax[0].hist(np.array(er_degree_values).flat, bins = 25,
           color = "red", edgecolor = "white")
ax[0].axvline(x=99*0.1, color='black', linestyle='--',
              linewidth=2, label='E(k)')
ax[0].set_title("Degree Distribution - 100 ER Networks")
ax[0].set_xlabel("Degree")
ax[0].set_ylabel("Frequency")
ax[0].set_box_aspect(1)
ax[0].legend()

# Edge distribution.
ax[1].hist(er_m, bins = 15,
           color = "red", edgecolor = "white")
ax[1].axvline(x=0.5*100*99*0.1, color='black', linestyle='--',
              linewidth=2, label='E(m)')
ax[1].set_title("Number of Links - 100 ER Networks")
ax[1].set_xlabel("Number of Links")
ax[1].set_ylabel("Frequency")
ax[1].set_box_aspect(1)
ax[1].legend()

plt.show()

While the Erdős-Rényi model fails to capture real degree distributions, it automatically encodes the small-world property.

In fact, it is possible to show inductively that the average number of nodes located at distance $s$ from a source node is equal to $\langle k \rangle^s$.

Therefore, assuming an Erdős-Rényi network has diameter $d$, it will be the case that
$$ \langle k \rangle^d \approx n \Rightarrow d = \frac{\ln(n)}{\ln(\langle k \rangle)}$$

The size of the largest connected component in an Erdős-Rényi network depends on the probability of having an edge between any two nodes.

More specifically, when $p = 0$, the network consists of a set of small isolated components that do not depend on the size of the network.

By contrast, if $p = 1$, the network acts a single $n$-clique that does depend on the size of the network.

In [ ]:
def sample_gnk(n, k, directed=False, loops=False):
  # Use an alternative parametrization for the Erdős-Rényi model.
  pk = k/(n-1)
  g = ig.Graph.Erdos_Renyi(n = n, p = pk, directed = directed, loops = loops)
  return g

In [ ]:
random.seed(10) # For reproducibility.

results = []

for k in np.arange(0,3.1,0.25):
  for r in range(10):
    # Store information about the largest connected component of each sampled network.
    tmp_g = sample_gnk(n = 100, k = k)
    rel_size_lcc = tmp_g.components().giant().vcount()/tmp_g.vcount()
    p = round(k/(99),ndigits=2)

    results.append({
        "k":k,
        "p": p,
        "replicate": r,
        "rel_size_lcc": rel_size_lcc
    })


# Store the results in a dataframe.
df_results = pd.DataFrame(results)
print(df_results.head())

In [ ]:
df_agg = df_results.groupby("k").agg({
    "rel_size_lcc": ["mean", "std"]
}).reset_index()

print(df_agg)

In [ ]:
# Plot the relationship between the expected degree and the relative size of the largest connected component.
fig, ax = plt.subplots(figsize=(5, 4))

ax.errorbar(df_agg["k"], df_agg["rel_size_lcc"]["mean"],
             yerr=df_agg["rel_size_lcc"]["std"],
             fmt='-o', label="Average S")
ax.axhline(y=0.25, color='red', linestyle='--',
              linewidth=1),
ax.axhline(y=0.8, color='red', linestyle='--',
              linewidth=1),
ax.set_xlabel("<k> (Expected Degree)")
ax.set_ylabel("S (Relative Size of LCC)")
ax.legend()

plt.show()

Overall, for an Erdős-Rényi network, it is possible to prove that, if $S$ is the proportion of nodes belonging to the largest connected component, the following relation holds:
$$S = 1 - e^{-S\langle k \rangle} \Rightarrow \langle k \rangle = \frac{\ln(1 - S)}{S}$$

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ax.errorbar(df_agg["k"], df_agg["rel_size_lcc"]["mean"],
             yerr=df_agg["rel_size_lcc"]["std"],
             fmt='-o', label="Average S")
ax.plot(theoretical_k,theoretical_s, label="Theoretical", color="blue")
ax.set_xlabel("<k> (Expected Degree)")
ax.set_ylabel("S (Relative Size of LCC)")
ax.legend()

plt.show()

The Erdős-Rényi model can also be defined using a $G(n, m)$ ensemble that fixes the number of nodes $n$ and the number of edges $m$, making it useful to create reference systems in which edge counts must remain constant.

In [ ]:
random.seed(10) # For reproducibility.

# Create four networks using the Erdős-Rényi model, choosing n = 25 and a parameter m.
m = round(0.1*24*24/2)
er1 = ig.Graph.Erdos_Renyi(n = 25, m = m)
er2 = ig.Graph.Erdos_Renyi(n = 25, m = m)
er3 = ig.Graph.Erdos_Renyi(n = 25, m = m)
er4 = ig.Graph.Erdos_Renyi(n = 25, m = m)

# Plot the generated networks.
fig, ax = plt.subplots(2,2,figsize=(9, 8))

ig.plot(
    er1,
    target=ax[0,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,0].set_title("First Graph")

ig.plot(
    er2,
    target=ax[0,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,1].set_title("Second Graph")

ig.plot(
    er3,
    target=ax[1,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,0].set_title("Third Graph")

ig.plot(
    er4,
    target=ax[1,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,1].set_title("Fourth Graph")

plt.tight_layout()
plt.show()

## The Watts-Strogatz Model

The Watts-Strogatz model aims to overcome a key limitation of the Erdős-Rényi model, whose low transitively does not reflect the behaviour observed in many real networks.

In particular, a Watts-Strogatz network is built starting from a regular ring lattice with $n$ nodes and, while each node is initially connected to a fixed number $k_n$ of nearest neighbours, each edge can be randomly rewired towards another node with probability $p$.

In [ ]:
random.seed(10) # For reproducibility.

# Create a Watts-Strogatz network with 1 dimension containing 20 nodes.
# Pick p = 0 as the rewiring probability.
g_ws = ig.Graph.Watts_Strogatz(
    dim=1,size=20,nei=2,p=0)
layout_g_ws = g_ws.layout_kamada_kawai() # Visualize the network using the Kamada-Kawai layout algorithm.

# Plot the resulting network.
fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_ws,
    target=ax,
    vertex_size=10,
    vertex_color="lightblue",
    edge_width = 1,
    layout = layout_g_ws
    )
ax.set_box_aspect(1)

In [ ]:
random.seed(10) # For reproducibility.

# Create a Watts-Strogatz network with 1 dimension containing 20 nodes.
# Pick p = 0.05 as the rewiring probability.
g_ws_rw = ig.Graph.Watts_Strogatz(
    dim=1,size=20,nei=2,p=0.05)

# Plot the resulting network.
fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_ws_rw,
    target=ax,
    vertex_size=10,
    vertex_color="lightblue",
    edge_width = 1,
    layout = layout_g_ws
    )
ax.set_box_aspect(1)

The structure of a Watts-Strogatz network implies that its degree distribution ranges between two edge cases:
1. If $p = 0$, the network follows a uniform degree distribution with mean degree $\langle k \rangle = 2k_n$.
2. If $p = 1$, the network exhibits the Poisson approximation of the Erdős-Rényi model.

In [ ]:
# Plot degree distributions for various Watts-Strogatz networks with varying parameter p.
fig, ax = plt.subplots(2,2,figsize=(9, 8))

# Pick p = 0.
ax[0,0].hist(ig.Graph.Watts_Strogatz(dim=1,size=20,nei=2,p=0.00).degree(),
             bins = range(1,11), color = "red", edgecolor = "white")
ax[0,0].set_xlabel("Degree")
ax[0,0].set_ylabel("Frequency")
ax[0,0].set_title("WS - p = 0")
ax[0,0].set_box_aspect(1)

# Pick p = 0.05.
ax[0,1].hist(ig.Graph.Watts_Strogatz(dim=1,size=20,nei=2,p=0.05).degree(),
             bins = range(1,11), color = "red", edgecolor = "white")
ax[0,1].set_xlabel("Degree")
ax[0,1].set_ylabel("Frequency")
ax[0,1].set_title("WS - p = 0.05")
ax[0,1].set_box_aspect(1)

# Pick p = 0.1.
ax[1,0].hist(ig.Graph.Watts_Strogatz(dim=1,size=20,nei=2,p=0.1).degree(),
             bins = range(1,11), color = "red", edgecolor = "white")
ax[1,0].set_xlabel("Degree")
ax[1,0].set_ylabel("Frequency")
ax[1,0].set_title("WS - p = 0.1")
ax[1,0].set_box_aspect(1)

# Pick p = 0.5.
ax[1,1].hist(ig.Graph.Watts_Strogatz(dim=1,size=20,nei=2,p=0.5).degree(),
             bins = range(1,11), color = "red", edgecolor = "white")
ax[1,1].set_xlabel("Degree")
ax[1,1].set_ylabel("Frequency")
ax[1,1].set_title("WS - p = 0.5")
ax[1,1].set_box_aspect(1)

plt.tight_layout()
plt.show()

It is possible to show that, for a Watts-Strogatz network such that $n \gg \langle k \rangle \gg \ln(n) \gg 1$, the small-world property can coexist with node cohesiveness.

In fact, for $p \to 0$, the network's average path length and global clustering coefficients are given by
$$L(p) \approx \frac{n}{2\langle k \rangle} \gg 1 \quad\quad C(p) \approx \frac{3}{4}$$

By contrast, as $p \to 1$, these metrics become closer to those of a random network as one has
$$L(p) \approx \frac{\ln(n)}{\ln(\langle k \rangle)} \quad\quad C(p) \approx \frac{\langle k \rangle}{n} \ll 1$$

In [ ]:
# Choose 15 log-spaced probabilities.
logspaced_values = np.logspace(np.log10(0.0001), np.log10(1), num=15)
print("A list of 15 bars logarithmically spaced \n", logspaced_values, "\n")

# Analyse the clustering coefficient of a Watts-Strogatz network with p = 0.
norm_cc = ig.Graph.Watts_Strogatz(
    dim=1,size=1000,nei=5,p=0).transitivity_avglocal_undirected()
print("The clustering coefficient for the given lattice at p = 0 ", norm_cc,"\n")

# Average the average path length of a Watts-Strogatz network with p = 0.
norm_l = ig.Graph.Watts_Strogatz(
    dim=1,size=1000,nei=5,p=0).average_path_length()
print("The average path length for the given lattice at p = 0 ", norm_l)

In [ ]:
random.seed(10) # For reproducibility.

# Store the clustering coefficient and average path length for each possible value of p.
cc_values = []
l_values = []

for i in logspaced_values:
  ws_tmp = ig.Graph.Watts_Strogatz(dim=1,size=1000,nei=5,p=i)

  cc_values.append(ws_tmp.transitivity_avglocal_undirected())
  l_values.append(ws_tmp.average_path_length())

In [ ]:
# Plot the values compared to the model with p = 0.
fig, ax = plt.subplots(figsize=(5, 4))

ax.scatter(logspaced_values, np.array(cc_values)/norm_cc, label="C(p) / C(0)", color="blue")
ax.scatter(logspaced_values, np.array(l_values)/norm_l, label="L(p) / L(0)", color="red", marker = "v")
ax.set_xscale('log')
ax.set_xlabel("Rewiring Probability (p)")
ax.legend()

plt.show()

## The Barabási-Albert Model

The Barabási-Albert model allows to generate scale-free networks through a probabilistic mechanism.

Assuming a network starts with $m_0$ nodes and arbitrarily chosen edges, provided each node has at least one link, the Barabási-Albert model develops the network through an iterative mechanism based on growth and preferential attachment.

At each step, a new node with $m \leq m_0$ links is added to the network and the probability that a link of the new node connects to node $i$, which has degree $k_i$, is given by
$$\pi(k_i) = \frac{k_i}{\sum_{j = 1}^n k_j}$$

Therefore, after $t$ steps, the Barabási-Albert model generates a network with $n = t + m_0$ nodes and $(m_0 + mt)$ total edges.

In [ ]:
random.seed(10) # For reproducibility.

# Create a Barabási-Albert network with n = 10000 nodes and m = 2 edges per new node.
g_ba = ig.Graph.Barabasi(n = 10000, m = 2, directed = True)

# Analyse the summary of the created network.
print(g_ba.summary())

# Analyse the frequency and degree distribution of the network.
print(degree_distribution(g_ba)["frequency"].head())
print(degree_distribution(g_ba)["distribution"].head())

For sufficiently large network sizes, the degree distribution of the Barabási-Albert model follows a power law with exponent $\gamma = 3$, meaning that $p(k) = \alpha k^{-3}$.

Moreover, by construction, the average degree of the resulting network is always equal to $\langle k \rangle = 2m$.

In [ ]:
# Plot the degree distribution of the Barabási-Albert network.
fig, ax = plt.subplots(1,2,figsize=(9, 4))

# Linear-linear scale.
ax[0].scatter(degree_distribution(g_ba)["frequency"].index,
degree_distribution(g_ba)["frequency"].values, color = "red")
ax[0].set_title("Degree Distribution (Lin-Lin)")
ax[0].set_xlabel("Degree")
ax[0].set_ylabel("Frequency")
ax[0].set_box_aspect(1)

# Log-log scale.
ax[1].scatter(degree_distribution(g_ba)["frequency"].index,
degree_distribution(g_ba)["frequency"].values, color = "red")
ax[1].set_title("Degree Distribution (Log-Log)")
ax[1].set_xlabel("Degree")
ax[1].set_ylabel("Frequency")
ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_box_aspect(1)

In particular, the coefficient of the distribution can be estimated using linear regression, although this approach may introduce a bias because the low-frequency region is located in the tail of the distribution.

In [ ]:
logx = np.log10(degree_distribution(g_ba)["frequency"].index)
logy_1 = np.log10(degree_distribution(g_ba)["frequency"].values+1)

# Find the slope and intercept of the regression model.
a, b = np.polyfit(logx, logy_1, 1)
print(a,b)

a_cut, b_cut = np.polyfit(logx[1:100], logy_1[1:100], 1)
print(a_cut,b_cut)

# Plot the resulting fits.
fig, ax = plt.subplots(figsize=(5, 4))#

ax.scatter(degree_distribution(g_ba)["frequency"].index,
degree_distribution(g_ba)["frequency"].values, color = "red")
ax.plot(10**logx, 10**(a*logx+b), color="grey", label= f"fit: y={a:.2f}x+{b:.2f}")
ax.plot(10**logx, 10**(a_cut*logx+b_cut), color="black", label= f"fit: y={a_cut:.2f}x+{b_cut:.2f}")
ax.set_ylim(0.5,)
ax.set_title("Degree Distribution (Log-Log)")
ax.set_xlabel("Degree")
ax.set_ylabel("Frequency")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_box_aspect(1)
ax.legend()

For this reason, the coefficient is typically estimated using the logarithmic binning method, which groups observations into bins whose sizes increase exponentially.

In particular, it can be shown that the estimated coefficient $\hat{\alpha}$ is related to the true coefficient $\alpha$ by the relationship $\hat{\alpha} = \alpha - 1$.

In [ ]:
# Implement linear binning using the np.linspace() function.
u = np.linspace(start = 1, stop = 10000, num = 5)
v = np.arange(len(u))

# Plot the resulting bins.
fig, ax = plt.subplots(figsize=(5, 4))

ax.plot(u, v, drawstyle='steps-post', color='black', linewidth=2)
ax.scatter(u, v, color='red')
ax.set_xlabel("u (Linear Steps)")
ax.set_ylabel("v (Linear Steps)")
ax.set_title("Linear Plot with Steps of \n Linearly Spaced Values")
ax.set_xticks(u)
ax.set_yticks(v)

In [ ]:
# Implement logarithmic binning using the np.logspace() function.
x = np.logspace(0, 4, num=5) # This returns [1, 10, 100, 1000, 10000].
x = np.logspace(np.log10(1), np.log10(10000), num = 5) # For better compatibiity with the np.linspace function.
y = np.arange(len(x))

# Plot the resulting bins.
fig, ax = plt.subplots(1,2,figsize=(9, 4))

ax[0].plot(x, y, drawstyle='steps-post', color='black', linewidth=2)
ax[0].scatter(x, y, color='red')
ax[0].set_xlabel("x (Log Spaced, Linearly Plotted)")
ax[0].set_ylabel("y (Linear Steps)")
ax[0].set_title("Linear Plot with Steps of \n Logarithmically Spaced x Values")
ax[0].set_xticks(x)
ax[0].set_yticks(y)

ax[1].plot(x, y, drawstyle='steps-post', color='black', linewidth=2)
ax[1].scatter(x, y, color='red')
ax[1].set_xscale('log')
ax[1].set_xlabel("x (Log Spaced, Logarithmically Plotted)")
ax[1].set_ylabel("y (Linear Steps)")
ax[1].set_title("Logarithmic Plot with Steps of \n Logarithmically Spaced x Values")
ax[1].set_xticks(x)
ax[1].set_yticks(y)

plt.show()

In [ ]:
# Create 11 bin edges spanning from the minimum to the maximum node degree of the network.
bin_edges = 10**np.linspace(
  np.log10(min(degree_distribution(g_ba)["frequency"].index)),
  np.log10(max(degree_distribution(g_ba)["frequency"].index)),
  num = 11)
bin_edges = bin_edges.round() # Round the values because degrees have to be integers.

# Place each edge into the corresponding logarithmic bin and group by bin.
binned_degrees = pd.cut(degree_distribution(g_ba)["frequency"].index,
                        bins=bin_edges, include_lowest=True)

binned_counts = (
    pd.DataFrame({
        "degree": degree_distribution(g_ba)["frequency"].index,
        "count": degree_distribution(g_ba)["frequency"].values,
        "bin": binned_degrees
    })
    .groupby("bin")["count"]
    .sum()
)

In [ ]:
# Perform a first fit using the logarithmic binning technique.
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(range(len(binned_counts.index)),
binned_counts.values, color = "red")
ax.set_title("Degree Distribution (Log-Logbinned)")
ax.set_xlabel("Logarithmic Bins of Degree")
ax.set_xticks(range(len(binned_counts.index)))
ax.set_xticklabels(binned_counts.index.astype(str), rotation=45, ha="right")
ax.set_ylabel("Frequency")
ax.set_yscale("log")
ax.set_box_aspect(1)

In [ ]:
# Run a linear regression fit using the actual values instead of the linear range.
[i.mid for i in binned_counts.index]
[i.right for i in binned_counts.index]

x_bins_mid = [i.mid for i in binned_counts.index]
x_bins_right = [i.right for i in binned_counts.index]

# Find intercept and slope of the linear regression fits.
a_bins_m, b_bins_m = np.polyfit(np.log10(x_bins_mid), np.log10(binned_counts.values), 1) # Using the midpoints.
print(a_bins_m,b_bins_m) # log10(y) = alog(x) + b.

a_bins_r, b_bins_r = np.polyfit(np.log10(x_bins_right), np.log10(binned_counts.values), 1) # Using the right side points.
print(a_bins_r,b_bins_r) # log10(y) = alog(x) + b.

# Plot the linear regression model.
fig, ax = plt.subplots(figsize=(5, 4))

ax.scatter(np.log10(x_bins_right), binned_counts.values, color = "red")
ax.plot(np.log10(x_bins_mid), 10**(a_bins_m*np.log10(x_bins_mid)+b_bins_m),
        color="grey", label= f"fit: y={a_bins_m:.2f}x+{b_bins_m:.2f}")
ax.plot(np.log10(x_bins_right), 10**(a_bins_r*np.log10(x_bins_right)+b_bins_r),
        color="black", label= f"fit: y={a_bins_r:.2f}x+{b_bins_r:.2f}")
ax.set_title("Degree Distribution (Log-Logbinned)")
ax.set_xlabel("Logarithmic Bins of Degree")
ax.set_xticks(np.log10(x_bins_right))
ax.set_xticklabels(binned_counts.index.astype(str), rotation=45, ha="right")
ax.set_ylabel("Frequency")
ax.set_yscale("log")
ax.set_box_aspect(1)
ax.legend()

In [ ]:
# The problem of the previous fit lies in the fact that the logarithmic bins are not evenly spaced.
print(np.log10(x_bins_right))
print(np.diff(np.log10((x_bins_right))))

# Convert the bins to ensure proportionality between the two axes.
axis_conversion = ((np.log10(816) - np.log10(2)) / (10 - 1))
print(axis_conversion)

# Scale the slope of the linear regression model.
print(-1.4139546860281622 * axis_conversion)
np.log10(816) - np.log10(2) # No Need to scale the intercept as well.

In [ ]:
a_bins_y, b_bins_y = np.polyfit(np.log10(x_bins_right), np.log10(binned_counts.values/binned_counts.values.sum()), 1)
print(a_bins_y,b_bins_y) # log(y) = alog(x) + b.

# Plot the actual linear regression with logarithmic binning.
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(np.log10(x_bins_right),
           binned_counts.values/binned_counts.values.sum(), color = "red")
ax.plot(np.log10(x_bins_right), 10**(a_bins_y*np.log10(x_bins_right)+b_bins_y),
        color="black", label= f"fit: y={a_bins_y:.2f}x+{b_bins_y:.2f}")
ax.set_title("Degree Distribution (Log-Logbinned)")
ax.set_xlabel("Logarithmic Bins of Degree")
ax.set_xticks(np.log10([i.right for i in binned_counts.index]))
ax.set_xticklabels(binned_counts.index.astype(str), rotation=45, ha="right")
ax.set_ylabel("Frequency")
ax.set_yscale("log")
ax.set_box_aspect(1)
ax.legend()

Generally speaking, the diameter of a Barabási-Albert network is given by
$$d_{\max} = \frac{\ln(n)}{\ln(\ln(n))}$$

This implies that the Barabási-Albert model exhibits the "ultra-small-world" property because distances tend to to be smaller compared to an Erdős-Rényi network of similar size.

Additionally, for moderate network sizes, a Barabási-Albert network tends to be locally more clustered than an Erdős-Rényi network as its global clustering coefficient is given by
$$C \approx \frac{\ln^2(n)}{n}$$

### Simulations of the Preferential Attachment Mechanism

In [ ]:
# Create a bag made up of the node IDs that appear in the edgelist.
ba_bag = np.array(g_ba.get_edgelist()).flatten()

random.seed(10)

edgelist_sample = []
for i in range(1000):
  # Generate random samples from the edge list.
  edgelist_sample.append(random.choice(ba_bag))

random_sample = random.sample(range(g_ba.vcount()),1000) # Sample node IDs without replacement.

In [ ]:
# Plot the resulting node and edge distributions.
fig, ax = plt.subplots(figsize=(5, 4))

ax.hist(g_ba.degree(edgelist_sample), edgecolor = "red", histtype='step', bins = 50, label = "Edgelist Sample")
ax.hist(g_ba.degree(random_sample), edgecolor = "blue", histtype='step', bins = 10, label = "Nodes Sample")
ax.set_xlabel("Degree")
ax.set_ylabel("Frequency")
ax.set_yscale("log")
ax.set_box_aspect(1)
ax.legend()